<a href="https://colab.research.google.com/github/EunjeLee0812/Sanhak/blob/seowonryeol/code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 라이브러리 설치
!pip install faster-whisper rapidfuzz g2pk konlpy python-mecab-ko

In [2]:
# #Googledrive 마운트(Colab 사이트 사용 시 주석 해제)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import torch, gc, sys
import os, re, json, glob, csv, random, glob, time
from dataclasses import dataclass
from typing import Dict, List, Optional, Any, Tuple
from g2pk import G2p
from faster_whisper import WhisperModel
from rapidfuzz.distance import Levenshtein
from rapidfuzz import process, fuzz
from mecab import MeCab
import importlib

In [4]:
# #Googledrive 마운트(Colab 사이트 사용 시 주석 해제)
# from google.colab import drive
# drive.mount('/content/drive')

In [5]:
# 1. 파일들이 위치한 경로를 시스템 경로에 추가
BASE_PATH = "/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results/"
%cd {BASE_PATH}

# 모듈 Import
from config.settings import *
from utils.normalizer import TextNormalizer
from utils.data_loader import load_transcripts
from utils.metrics import calculate_cer, calculate_wer, evaluate_proper_nouns
from core.asr_engine import ASR
from core.bias_manager import BiasManager
from core.post_processor import postprocess_with_hotwords


/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results


In [7]:
#그래픽카드 메모리 남용을 막기 위한 캐시 초기화

gc.collect()
torch.cuda.empty_cache()

# 1-5. 결과 저장 경로[현재 시간 반영해서 파일별 구분 용이]
#results 폴더 없으면 생성
if not os.path.exists(os.path.join(BASE_PATH,"result")):
    os.makedirs(os.path.join(BASE_PATH,"result"))

now = time.gmtime(time.time()+(9*3600)) #한국 시간
formatted = time.strftime("[%Y%m%d_%H%M]", now)
OUT_ROWS = f"./results/{formatted}_asr_detail.csv"
OUT_SUM  = f"./results/{formatted}_asr_summary.csv"

def summarize(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    agg: Dict[Tuple[int, int, int], Dict[str, Any]] = {}

    for r in rows:
      # [수정 2] 안전한 Key 접근 (.get 사용)
        # hotwords_strategy가 없는 옛날 데이터라면 기본값 1을 부여
        hotwords_strategy = r.get("hotwords_strategy", "random")
        bias_update_cycle = r.get("bias_update_cycle", 1)

        key = (int(r["top_k"]), int(r["preprocess_on"]), hotwords_strategy, bias_update_cycle)
        a = agg.setdefault(key, {"top_k": key[0], "preprocess_on": key[1], "hotwords_strategy":key[2], "bias_update_cycle":key[3], "files_num": 0, "cer_sum": 0.0, "wer_sum": 0.0, "pn_recall_sum": 0.0, "pn_cer_sum": 0.0})
        a["files_num"] += 1
        a["cer_sum"] += float(r["cer"])
        a["wer_sum"] += float(r["wer"])
        if r.get("pn_recall") is not None:
            a["pn_recall_sum"] += float(r["pn_recall"])
            a["pn_cer_sum"] += float(r["pn_cer"])

    out = []
    for k, a in sorted(agg.items()):
        out.append({
            "top_k": a["top_k"],
            "preprocess_on": a["preprocess_on"],
            "hotwords_strategy": a["hotwords_strategy"],
            "bias_update_cycle": a["bias_update_cycle"],
            "used_files_num": AUDIO_FILE_MAX,
            # round(값, 4)를 통해 소수점 4자리까지 반올림합니다.
            "cer_avg": round(a["cer_sum"] / max(1, a["files_num"]), 4),
            "wer_avg": round(a["wer_sum"] / max(1, a["files_num"]), 4),
            "pn_recall_avg": round(a["pn_recall_sum"] / max(1, a["files_num"]), 4),
            "pn_cer_avg": round(a["pn_cer_sum"] / max(1, a["files_num"]), 4)
        })
    return out

# ==============================================================================
# 메인 실행 로직
# ==============================================================================

# 1. 초기화 및 로드
if not os.path.exists(BASE_DIR):
    print("[WARN] Base path not found. Checking local..")

normalizer = TextNormalizer()
mecab = MeCab()
bias_mgr = BiasManager(BIAS_PATH)
transcripts = load_transcripts(TRANSCRIPTS_PATH)
files = glob.glob(os.path.join(AUDIO_FOLDER, "**/*.mp4"), recursive=True)[:AUDIO_FILE_MAX]

# ASR 모델 로드
asr = ASR(ASR_MODEL, ASR_DEVICE, ASR_COMPUTE,initial_prompt=KOREAN_ONLY_PROMPT)

rows: List[Dict[str, Any]] = []  # [수정] 결과 데이터를 저장할 리스트

# 2. 실험 루프
for top_k in HOTWORD_TOPK_SWEEP:
    # 전략별 테스트 (1: Random, 2: Hybrid)
    for hotwords_strategy in HOTWORD_STRATEGY_SWEEP:
        if RESET_BIASING_LIST : bias_mgr.reset_biasing_list(BIAS_PATH)
        #bias_update_cycle 주기마다 biasing_lists 업데이트
        for bias_update_cycle in BIAS_ITERATION_CYCLE_SWEEP:
            #repeat_cnt : 학습 반복횟수 (고유명사 인식 가중치가 계속 누적되다가 bias_update_cycle번 후 누적된 가중치 업데이트)
            for repeat_cnt in range(bias_update_cycle):
              current_hotwords = bias_mgr.get_weighted_hotwords(top_k, mode=hotwords_strategy)

              for pp_on in POSTPROCESS_SWEEP:
                  print(f"\n[RUN] Top-K: {top_k} | Iteration: {repeat_cnt}/{bias_update_cycle} | PostProcess: {pp_on}")
                  print(f"hotwords : {current_hotwords}\n")

                  for audio_path in files:
                      fname = os.path.basename(audio_path)
                      meta = transcripts.get(fname, {"text": "", "entities": []})

                      # 1) ASR 추론
                      hyp_raw = asr.transcribe(audio_path, "ko", ASR_BEAM, hotwords=current_hotwords)

                      # 2) 후처리 (ON일 때만)
                      if pp_on:
                          hyp_final, replog = postprocess_with_hotwords(
                              hyp_raw, current_hotwords, normalizer, # normalizer 주입
                              gate=RULE_GATE, tol=RULE_TOL, wratio_th=RULE_WRATIO_TH
                          )
                      else:
                          hyp_final, replog = hyp_raw, []

                      # 3) 평가
                      cer = calculate_cer(meta["text"], hyp_final, normalizer)
                      wer = calculate_wer(meta["text"], hyp_final, normalizer, mecab)
                      pn_recall, pn_cer, hyp_ents = evaluate_proper_nouns(meta["entities"], hyp_final, normalizer)

                      # 4) 학습 기록 (Bias Manager)
                      bias_mgr.add_hit(hyp_ents)
                      pn_recall = pn_recall if pn_recall is not None else 0.0

                      # 로그 출력

                      print(f"- file: {os.path.dirname(audio_path).split("/")[-1]}/{fname} | pp_on(후처리 여부)={pp_on} | cer={cer:.4f} | wer={wer:.4f} | pn_cer(고유명사 cer)={pn_cer} | pn_recall(고유명사 recall)={pn_recall:.4f}")
                      print(f"ref_text:  [{meta["text"]}]\nhyp_raw:   [{hyp_raw}]\nhyp_final: {hyp_final}\nref_text_pn: {meta["entities"]}\nhyp_pn:     {hyp_ents}\n")


                      # [수정] 결과를 rows 리스트에 저장
                      rows.append({
                          "file": f'{os.path.dirname(audio_path).split("/")[-1]}/{fname}',
                          "top_k": top_k, "preprocess_on": pp_on, "hotwords_strategy" : "random" if hotwords_strategy==1 else "hybrid",
                          "hotwords": current_hotwords, "bias_update_cycle": bias_update_cycle,
                          "cer": f"{cer:.4f}", "wer": f"{wer:.4f}", "pn_recall": f"{pn_recall:.4f}", "pn_cer": f"{pn_cer:.4f}",
                          "ref_text": meta["text"], "hyp_raw": normalizer.normalize(hyp_raw), "hyp_final": hyp_final,
                          "ref_text_pn": meta["entities"], "hyp_pn": hyp_ents,
                          "replog": json.dumps(replog, ensure_ascii=False)
                        })

        # 3. 최종 학습 반영 및 저장
        bias_mgr.finalize(bias_update_cycle)

with open(OUT_ROWS, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

summary = summarize(rows)
with open(OUT_SUM, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(summary[0].keys()))
    w.writeheader()
    w.writerows(summary)

print("\n[DONE] All experiments finished.")

[SUCCESS] /content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results/lists/biasing_list.json 데이터가 모두 0으로 초기화되었습니다.

[RUN] Top-K: 20 | Iteration: 0/3 | PostProcess: 1
hotwords : ['킹메이커', '닥터스트레인지', '달짝지근해', '이정재', '스트릿우먼파이터', '삼식이삼촌', '백종원', '글래디에이터이', '크로스', '김준현', '차린건쥐뿔도없지만', '엠비씨', '비상선언', '졸업', '안판석', '선재업고튀어', '부활남', '오징어게임이', '전란', '티브이엔']

- file: lee-minseon/record0.mp4 | pp_on(후처리 여부)=1 | cer=0.1071 | wer=0.1765 | pn_cer(고유명사 cer)=0.15 | pn_recall(고유명사 recall)=0.5000
ref_text:  [넷플릭스에서 이사랑도통역이되나요 내가 보던 부분에서 재생해줘]
hyp_raw:   [넷플릭스에서 이 사랑도 통해하게 되나요? 내가 보던 부분에서 재생해줘]
hyp_final: 넷플릭스에서 이 사랑도 통해하게 되나요? 내가 보던 부분에서 재생해줘
ref_text_pn: ['넷플릭스', '이사랑도통역이되나요']
hyp_pn:     ['넷플릭스', '이사랑도통해하게되나요']

- file: lee-minseon/record11.mp4 | pp_on(후처리 여부)=1 | cer=0.0909 | wer=0.0769 | pn_cer(고유명사 cer)=0.25 | pn_recall(고유명사 recall)=0.5000
ref_text:  [티브이엔 유퀴즈온더블럭 재방송 언제 하는지 찾아줘]
hyp_raw:   [티비엔 유퀴즈 온 더 블럭 재방송 언제 하는지 찾아줘]
hyp_final: 티비엔 유퀴즈 온 더 블럭 재방송 언제 하는지 찾아줘
ref_text_pn: ['티브이엔', '유퀴즈온더블럭']
hyp_pn:   